# Left/right choice bias


In [1]:
# Data handling
import os
import re
import csv
import warnings
import itertools as it

import numpy as np
import pandas as pd
from tqdm import tqdm

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
from matplotlib.colors import Normalize
from matplotlib.patches import FancyArrowPatch

# Networks and statistical tests
import networkx as nx
from scipy.stats import binomtest, chi2_contingency, chisquare
import statsmodels.stats.multitest as stm

# Functions stored in a1_helpers.py
from a1_helpers import (
    get_time_taken_sec,
    extract_strategies,
    get_percentages,
    parse_values,
    directed_to_undirected_with_summed_weights,
    graph_primer,
    create_dic,
    percentage_scaler,
    get_transition_df,
    Euclidean_Dist,
)

# Progress bars for pandas
tqdm.pandas()

# Hide deprecation warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
from pathlib import Path

# Run with the notebook's Analysis folder as the working directory.
assert Path('a1_helpers.py').is_file(), 'Set the working directory to the Analysis folder.'
fig_folder = '../Figure_paper/'
Path(fig_folder).mkdir(parents=True, exist_ok=True)
colors = ['#0033ff', '#ff4ff1']

In [3]:
sequence_df = pd.read_csv('../Combined_data/seq_df_2025.csv', index_col=0)
sequence_df['game'] = sequence_df['experiment'].str[-1]

In [4]:
toplot = sequence_df.groupby(['ppid','trial_num','scale','cond']).first().reset_index()
DATA_nodemo = toplot.copy()


DATA_nodemo.loc[:, 'L_R'] = DATA_nodemo['visits'].apply(parse_values)


proportions = DATA_nodemo.groupby(['cond', 'scale', 'L_R']).size().reset_index(name='count')
proportions['percentage'] = proportions.groupby(['cond', 'scale'])['count'].transform(lambda x: (x / x.sum()) * 100)

In [5]:
right = proportions.loc[proportions.L_R == 'R']
for cond,DF in right.groupby('cond'):
    print(cond)
    list_val  = list()
    list_scale= list()
    for scale,df in DF.groupby('scale'):
        list_scale.append(scale)
        list_val.append(np.round(df.percentage.values[0]))

    print(f'{list_scale[0]}: {list_val[0]}%, {list_scale[1]}: {list_val[1]}%, {list_scale[2]} : {list_val[2]}%, {list_scale[3]}: {list_val[3]}%, {list_scale[4]} : {list_val[4]}%')

C1
S1: 63.0%, S2: 60.0%, S3 : 57.0%, S4: 49.0%, S5 : 57.0%
C2
S1: 51.0%, S2: 62.0%, S3 : 52.0%, S4: 51.0%, S5 : 64.0%


In [6]:
left = proportions.loc[proportions.L_R == 'L']
for cond,DF in left.groupby('cond'):
    print(cond)
    list_val  = list()
    list_scale= list()
    for scale,df in DF.groupby('scale'):
        list_scale.append(scale)
        list_val.append(np.round(df.percentage.values[0]))

    print(f'{list_scale[0]}: {list_val[0]}%, {list_scale[1]}: {list_val[1]}%, {list_scale[2]} : {list_val[2]}%, {list_scale[3]}: {list_val[3]}%, {list_scale[4]} : {list_val[4]}%')

C1
S1: 37.0%, S2: 40.0%, S3 : 43.0%, S4: 51.0%, S5 : 43.0%
C2
S1: 48.0%, S2: 38.0%, S3 : 47.0%, S4: 48.0%, S5 : 36.0%


In [7]:
corrected_alpha = 0.05/3 
for cond, df_ in DATA_nodemo.groupby('cond') :
        print(cond)
        pvals =list()
        for name, group in df_.groupby('scale') :
                k = group.loc[group.L_R ==  'L'].shape[0]
                result = binomtest(k, n=group.shape[0], p=0.5)
                #print(name,result,result.pvalue<corrected_alpha)
                pvals.append(result.pvalue)

        _,pvals_correctedndarray,_,_= stm.multipletests(pvals, alpha=0.05, method='holm')
        print(pvals_correctedndarray)

C1
[0.07716182 0.26600174 0.26600174 1.         0.45525231]
C2
[1.         0.12567244 1.         1.         0.04316793]


In [8]:
C_df = DATA_nodemo.loc[DATA_nodemo.cond == 'C1']
k = C_df.loc[C_df.L_R ==  'R'].shape[0]
result = binomtest(k, n = C_df.shape[0], p=0.5,alternative = 'greater')
result

BinomTestResult(k=319, n=558, alternative='greater', statistic=0.5716845878136201, pvalue=0.0004048458210095829)

In [9]:
for config in ['C1','C2']:
    df = DATA_nodemo.loc[DATA_nodemo.cond == config]    

    # Replace these with actual total counts
    scales_list =['S1','S2','S3','S4','S5']
    observed = pd.DataFrame(index =scales_list )

    for scale in scales_list:
        group = df.loc[df.scale ==scale]    
        observed.loc[scale,['left']] = group.loc[group.L_R ==  'L'].shape[0]
        observed.loc[scale,['right']] = group.loc[group.L_R ==  'R'].shape[0]

    observed = observed.to_numpy()
    # Perform Chi-Square Test
    chi2, p, dof, expected = chi2_contingency(observed)
    print(config)
    # Output results
    print(f"Chi-Square Statistic: {chi2:.4f}")
    print(f"P-value: {p:.4f}")
    print(f"Degrees of Freedom: {dof}")

C1
Chi-Square Statistic: 3.5922
P-value: 0.4640
Degrees of Freedom: 4
C2
Chi-Square Statistic: 6.2747
P-value: 0.1795
Degrees of Freedom: 4
